## Imports

In [1]:
import os
import pandas as pd, math
import numpy as np
from pyhive import presto
from datetime import datetime, date, timedelta
import warnings

# Ignore a specific warning by its type
warnings.filterwarnings("ignore")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

## Connection

In [2]:
presto_port = '80'
# username = 'airflow-user'
username = 'manoj.ravirajan@rapido.bike'

connection1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
connection2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
connection3 = presto.connect('processing-2.processing.data.production.internal', presto_port, username)
connection4 = presto.connect('processing.processing.data.production.internal', presto_port, username)

## Parameter

In [3]:
city = 'Chennai'
date_list = ['2024-04-07', '2024-04-14', '2024-04-21', '2024-04-28', '2024-05-05']
base_week = date_list[0]

## Dataset & User-Defined Functions

In [4]:
def consideration_query(run_date, city_name, base_week):
    
    rr_data_query = f"""
  
        with dormant_customer as (
            
            select
                customer_id as customer,
                date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) rapido_first_ride,
                ceil(cast(date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) as real)/10)*10 rapido_age_seg_fr,
                
                taxi_lifetime_last_ride_date, taxi_lifetime_first_ride_date, 
                taxi_lifetime_stage, previous_taxi_lifetime_stage, taxi_income_segment,
                taxi_lifetime_rides, taxi_recency_segment, taxi_rr_active_days_last_21_days as run_taxi_rr_active_days_last_21_days,
                taxi_lifetime_last_ride_city city_name, 
                
                fe_mean_intent_type, fe_intent_trend_type, fe_potential_per_type, 
                fe_regularity_type, fe_recency_type, fe_volatility_type,
                
                gender_tag, ps_tag_link, ps_tag_auto,
                
                taxi_service_affinity, taxi_distance_preference, taxi_time_of_day_preference, taxi_day_of_week_preference
            
            from 
                datasets.iallocator_customer_segments
            where 
                date(run_date) = date('{base_week}')
                -- and taxi_lifetime_last_ride_date = date('2023-11-01')
                and taxi_recency_segment in ('RECENT')
                and taxi_lifetime_rides > 4
                and taxi_lifetime_last_ride_city in ('{city_name}')
        ),
        
        base as (
        
            select 
                a.*,
                case when coalesce(b.rr,0) <1 then 0 else 1 end as rr_retention_flag,
                case when coalesce(b.fe,0) <1 then 0 else 1 end as fe_retention_flag,
                case when coalesce(b.ao,0) <1 then 0 else 1 end as ao_retention_flag,
                case when coalesce(b.nr,0) <1 then 0 else 1 end as net_retention_flag,
                
                case when coalesce(b.nr_post_w01,0) <1 then 0 else 1 end as nr_flag_w01,
                case when coalesce(b.nr_post_w02,0) <1 then 0 else 1 end as nr_flag_w02,
                case when coalesce(b.nr_post_w03,0) <1 then 0 else 1 end as nr_flag_w03,
                case when coalesce(b.nr_post_w04,0) <1 then 0 else 1 end as nr_flag_w04,
                case when coalesce(b.nr_post_w05,0) <1 then 0 else 1 end as nr_flag_w05,
                case when coalesce(b.nr_post_w06,0) <1 then 0 else 1 end as nr_flag_w06,
                case when coalesce(b.nr_post_w07,0) <1 then 0 else 1 end as nr_flag_w07,
                case when coalesce(b.nr_post_w08,0) <1 then 0 else 1 end as nr_flag_w08,
                case when coalesce(b.nr_post_w09,0) <1 then 0 else 1 end as nr_flag_w09,
                case when coalesce(b.nr_post_w10,0) <1 then 0 else 1 end as nr_flag_w10,
                
                case when coalesce(b.rr_post_w01,0) <1 then 0 else 1 end as rr_flag_w01,
                case when coalesce(b.rr_post_w02,0) <1 then 0 else 1 end as rr_flag_w02,
                case when coalesce(b.rr_post_w03,0) <1 then 0 else 1 end as rr_flag_w03,
                case when coalesce(b.rr_post_w04,0) <1 then 0 else 1 end as rr_flag_w04,
                case when coalesce(b.rr_post_w05,0) <1 then 0 else 1 end as rr_flag_w05,
                case when coalesce(b.rr_post_w06,0) <1 then 0 else 1 end as rr_flag_w06,
                case when coalesce(b.rr_post_w07,0) <1 then 0 else 1 end as rr_flag_w07,
                case when coalesce(b.rr_post_w08,0) <1 then 0 else 1 end as rr_flag_w08,
                case when coalesce(b.rr_post_w09,0) <1 then 0 else 1 end as rr_flag_w09,
                case when coalesce(b.rr_post_w10,0) <1 then 0 else 1 end as rr_flag_w10,
                
                case when coalesce(b.fe_post_w01,0) <1 then 0 else 1 end as fe_flag_w01,
                case when coalesce(b.fe_post_w02,0) <1 then 0 else 1 end as fe_flag_w02,
                case when coalesce(b.fe_post_w03,0) <1 then 0 else 1 end as fe_flag_w03,
                case when coalesce(b.fe_post_w04,0) <1 then 0 else 1 end as fe_flag_w04,
                case when coalesce(b.fe_post_w05,0) <1 then 0 else 1 end as fe_flag_w05,
                case when coalesce(b.fe_post_w06,0) <1 then 0 else 1 end as fe_flag_w06,
                case when coalesce(b.fe_post_w07,0) <1 then 0 else 1 end as fe_flag_w07,
                case when coalesce(b.fe_post_w08,0) <1 then 0 else 1 end as fe_flag_w08,
                case when coalesce(b.fe_post_w09,0) <1 then 0 else 1 end as fe_flag_w09,
                case when coalesce(b.fe_post_w10,0) <1 then 0 else 1 end as fe_flag_w10

            from 
                dormant_customer a
            left join
                (
                select 
                    customerid,
                    -- sum(ao_sessions_unique_daily) ao,
                    -- sum(fe_sessions_unique_daily) fe,
                    -- sum(rr_sessions_unique_daily) rr,
                    -- sum(net_rides_daily) nr,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then ao_sessions_unique_daily end) as ao,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr,
                    
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w10,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w09,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w08,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w07,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w06,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w05,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w04,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w03,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w02,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w01,
                    
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w10,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w09,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w08,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w07,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w06,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w05,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w04,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w03,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w02,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w01,
                    
                    
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w10,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w09,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w08,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w07,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w06,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w05,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w04,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w03,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w02,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w01

                from 
                    datasets.customer_rf_daily_kpi 
                where 
                    date(day) > date('{base_week}') 
                    and date(day) <= date('{base_week}') + interval '70' day
                    and service_name in ('Link','Auto', 'CabEconomy')
                    and city in ('{city_name}')
                group by 1
                ) b
                on a.customer=b.customerid

        ),
    
        rr_data as (
        
            select 
                fe.*
            from
                (
                select 
                    aa.*, 
                    bb.rr_LT, 
                    bb.rapido_age,
                    bb.rapido_age_seg
                from
                    (
                    select 
                        customerid,
                        sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w13,
                        sum(case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w12,
                        sum(case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w11,
                        sum(case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w10,
                        sum(case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w09,
                        sum(case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w08,
                        sum(case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w07,
                        sum(case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w06,
                        sum(case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w05,
                        sum(case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w04,
                        sum(case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w03,
                        sum(case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w02,
                        sum(case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w01,
                        sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_91days,
                        sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_91days,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then day end) as active_rr_days_w13,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then day end) as active_rr_days_w12,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then day end) as active_rr_days_w11,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then day end) as active_rr_days_w10,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then day end) as active_rr_days_w09,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then day end) as active_rr_days_w08,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then day end) as active_rr_days_w07,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then day end) as active_rr_days_w06,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then day end) as active_rr_days_w05,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then day end) as active_rr_days_w04,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then day end) as active_rr_days_w03,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then day end) as active_rr_days_w02,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then day end) as active_rr_days_w01,
                        day((date('{run_date}') + interval '-01' day) - date(max(day))) as days_since_last_rr
                    from 
                        (
                        select 
                            customerid,day, 
                            max(fe_sessions_unique_daily) as fe_sessions_unique_daily,  
                            sum(rr_sessions_unique_daily) as rr_sessions_unique_daily
                        from 
                            datasets.customer_rf_daily_kpi a
                        join
                            base b
                            on a.customerid = b.customer
                            and 1 = 1 
                            and city = '{city_name}'
                            and rr_sessions_unique_daily >= 1 
                            and day >= date_format(date('{run_date}') + interval '-91' day , '%Y-%m-%d')
                            and day <= date_format(date('{run_date}') + interval '-1' day , '%Y-%m-%d')
                            and service_name in ('Link','Auto', 'CabEconomy')
                        group by customerid, day
                        )
                    group by customerid
                    ) aa 
                left join 
                    (
                    select 
                        customerid, 
                        sum(rr_sessions_unique_daily) as rr_LT , 
                        day((date('{run_date}') + interval '-01' day) - date(min(day))) as rapido_age,
                        ceil(cast(day((date('{run_date}') + interval '-01' day) - date(min(day))) as real)/10)*10 rapido_age_seg
                    from 
                        datasets.customer_rf_daily_kpi a
                    join
                        base b
                        on a.customerid=b.customer
                        and rr_sessions_unique_daily >= 1 
                        and city = '{city_name}'
                    group by 1
                    ) bb
                    on aa.customerid = bb.customerid
                ) fe 
        )
        
        select 
            a.*, b.*,
            '{run_date}' as run_date
        from 
            base a
        left join
            rr_data b
            on a.customer=b.customerid
    """

    try:
        data = pd.read_sql(rr_data_query, connection3)
    except:
        data = pd.read_sql(rr_data_query, connection4)
        
    
    ## Data fetched successfully
    print(data.shape)


    ## .mask(data[columns_to_replace] == 0, pd.NA) replaces all 0 values in these columns with pd.NA.
    columns_to_replace = ['active_rr_days_w13', 'active_rr_days_w12', 'active_rr_days_w11','active_rr_days_w10', 'active_rr_days_w09',
                          'active_rr_days_w08', 'active_rr_days_w07', 'active_rr_days_w06', 'active_rr_days_w05', 'active_rr_days_w04',
                          'active_rr_days_w03', 'active_rr_days_w02', 'active_rr_days_w01']
    data[columns_to_replace] = data[columns_to_replace].mask(data[columns_to_replace] == 0, pd.NA)
    

    ## Unused
    ## RR-Normalization = Total no. of RR/Active days
    """
    data['rr_norm_w13'] = data.rr_w13/data.active_rr_days_w13
    data['rr_norm_w12'] = data.rr_w12/data.active_rr_days_w12
    data['rr_norm_w11'] = data.rr_w11/data.active_rr_days_w11
    data['rr_norm_w10'] = data.rr_w10/data.active_rr_days_w10
    data['rr_norm_w09'] = data.rr_w09/data.active_rr_days_w09
    data['rr_norm_w08'] = data.rr_w08/data.active_rr_days_w08
    data['rr_norm_w07'] = data.rr_w07/data.active_rr_days_w07
    data['rr_norm_w06'] = data.rr_w06/data.active_rr_days_w06
    data['rr_norm_w05'] = data.rr_w05/data.active_rr_days_w05
    data['rr_norm_w04'] = data.rr_w04/data.active_rr_days_w04
    data['rr_norm_w03'] = data.rr_w03/data.active_rr_days_w03
    data['rr_norm_w02'] = data.rr_w02/data.active_rr_days_w02
    data['rr_norm_w01'] = data.rr_w01/data.active_rr_days_w01
    """
    
    raw_customer_data = data[data['customerid']!=0]
    back_up = raw_customer_data.copy() ## With NA as value
    fe_data2 = raw_customer_data.fillna(0)
    fe_data = fe_data2[fe_data2['customerid']!=0] ## With 0 as value
    
    
    ## Consistency/Regularity Threshold 
    ## RR Regularity → Count of active weeks in last 13 weeks/ (13 or Rapido age in weeks if a customer is less than 13 weeks old)
    ## Active Days Mean → Total active days in last 13 weeks/ count of active weeks in last 13 weeks
    
    ## [OLD version - Unused] 
    """
    def regularity_decider(x,active_mean):
    
        if x<=0.3:
            return '05_rare_need'
        elif x>0.3 and x<=0.5:
            return '04_monthly'
        elif x>0.5 and x<=0.75:
            return '03_bi_weekly'
        else:
            if active_mean>2.6:
                return '01_daily'
            else:
                return '02_weekly'
        # return 'weekly'
    """

    ## [Relaxed threshold]
    def regularity_decider_v2(x,active_mean):
        
        if x<=0.2:
            return '05_rare_need'
        elif x>0.2 and x<=0.4:
            return '04_monthly'
        elif x>0.4 and x<=0.689:
            return '03_bi_weekly'
        else:
            if active_mean>2.6:
                return '01_daily'
            else:
                return '02_weekly'


    ## Intent Threshold
    ## Intent → Total RR in last 13 weeks/ No. of Active weeks
    def intent_decider(x):
        
        if x<=2:
            return '03_Low'
        elif x>2 and x<=5:
            return '02_Medium'
        else:
            return '01_High'
    

    ## Consideration signal
    def consideration_signal_creator(x,y,active_mean):
        
        if y<=1.5:
            if x<=0.3:
                return 'rarely_1_rr'
            elif x>0.3 and x<=0.5:
                return 'once_in_4_weeks_1_rr'
            elif x>0.5 and x<=0.75:
                return 'once_in_2_weeks_1_rr'
            else:
                return 'weekly_1_rr'
        else:
            if x<=0.3:
                return 'rarely_2+_rr'
            elif x>0.3 and x<=0.5:
                return 'once_in_4_weeks_2+_rr'
            elif x>0.5 and x<=0.75:
                return 'once_in_2_weeks_2+_rr'
            else:
                return 'weekly_2+_rr'
                
                # Dead-code
                # if active_mean>2.6:
                #     return 'daily'
                # else:
                #     return 'weekly'


    ## Correction factor
    def correction_factor(rr_weight,rapido_age):
        
        if rapido_age >= 91:
            week_inactive = 0
            multiplier = 1

        elif rapido_age == 0:
            week_inactive = (91-rapido_age)//7 - 1
            multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2)))

        # 7,14,21,28,...
        elif rapido_age%7 == 0:
            week_inactive = (91-rapido_age)//7 - 1
            multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2)))    
            
        else:
            week_inactive = (91-rapido_age)//7
            multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2))) 
        
        return (multiplier, 
                week_inactive, 
                (rr_weight.sum()- (week_inactive*(week_inactive+1)/2)), 
                ((rr_weight*rr_weight).sum() - (week_inactive*(week_inactive+1)*(2*week_inactive+1)/6))
               )

    
    ## Recency trend
    def recency_type(days_since_last_fe):
        
        if days_since_last_fe <= 6: 
            return('Recent')
        elif days_since_last_fe <= 20:
            return('Stationary')
        else:
            return('Dormant')

    ## Intent trend
    def intent_trend_type(intent_trend):
        
        if intent_trend <= -0.1:
            return('03_Declining')
        elif intent_trend <= 0.1:
            return('02_Stable')
        else:
            return('01_Increasing')

    # Dead-code
    # def custom_func(mean_intent,rr_trend):
    #     if mean_intent=='High':
    #         return rr_trend
    #     elif mean_intent=='Low' and rr_trend=='01_Increasing':
    #         return '01_Increasing'
    #     else:
    #         return '02_Stable'

    
    
    ## Creating Final signal
    def create_L3_signals(fe_data, back_up): 
        
        print('Creating L3 Signal')
        rr_weight = np.array([1,2,3,4,5,6,7,8,9,10,11,12,13])

        ## sum (rr_sessions_unique_daily)
        fe_cols = ['rr_w13', 'rr_w12', 'rr_w11', 'rr_w10', 'rr_w09','rr_w08', 'rr_w07', 
                   'rr_w06','rr_w05', 'rr_w04', 'rr_w03', 'rr_w02','rr_w01']   
        ## count (rr_days)
        activedays_cols = ['active_rr_days_w13', 'active_rr_days_w12', 'active_rr_days_w11', 'active_rr_days_w10', 
                           'active_rr_days_w09','active_rr_days_w08', 'active_rr_days_w07', 'active_rr_days_w06',
                           'active_rr_days_w05', 'active_rr_days_w04', 'active_rr_days_w03','active_rr_days_w02', 
                           'active_rr_days_w01'] 
        
        ## Unused
        # norm_intent=['rr_norm_w13', 'rr_norm_w12', 'rr_norm_w11', 'rr_norm_w10','rr_norm_w09', 'rr_norm_w08', 'rr_norm_w07', 
        #              'rr_norm_w06','rr_norm_w05', 'rr_norm_w04', 'rr_norm_w03', 'rr_norm_w02','rr_norm_w01']
        # fe_data['fe2rr_91days'] = fe_data.apply(lambda x: (x['rr_91days']/x['fe_91days']) if x['fe_91days'] > 0 else 0.5,axis=1)
        
        
        print('Calctn-1')
        fe_data['rr_intent_multiplier'] = fe_data['rapido_age'].apply(lambda x: correction_factor(rr_weight,x)[0])
        ## Unused
        # fe_data['rr_need'] = np.array(fe_data[fe_cols]).max(axis=1)
        # fe_data['no_week_rr_done'] =  back_up[fe_cols].count(axis=1)
        # fe_data['%week_rr_done'] = fe_data.apply(lambda x: x['no_week_rr_done']*(1/(13 - correction_factor(rr_weight,x['rapido_age'])[1])), axis = 1 )
        fe_data['rr_mean_intent'] =  back_up[fe_cols].mean(axis=1) ## Check logic once

        ## Unused
        # fe_data['rr_mean_intent_norm'] =  back_up[norm_intent].mean(axis=1)
        # fe_data['rr_mean_intent_type'] = fe_data['rr_mean_intent'].apply(lambda x: 'Low' if x <= 3 else 'High')
        fe_data['rr_mean_intent_final'] = fe_data['rr_mean_intent'].apply(lambda x: intent_decider(x))

        # Dead-code
        # fe_data['rr_cov'] =  (np.sqrt(back_up[fe_cols].var(axis=1))/back_up[fe_cols].mean(axis=1))
        # fe_data['rr_volatility_type'] = fe_data.apply(lambda x: 'Low' if x['rr_cov'] <= 0.4 else 'High', axis = 1)
        # fe_data['rr_potential'] = fe_data['rr_need'] - fe_data['rr_mean_intent']
        # fe_data['rr_potential_per'] = (fe_data['rr_need'] - fe_data['rr_mean_intent'])/fe_data['rr_mean_intent']
        # fe_data['rr_potential_per_type'] = fe_data['rr_potential_per'].apply(lambda x: 'High' if x >= 0.5 else 'Low')
        
        ## Unused
        # fe_data['rr_trend_n'] = (((13 - fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[1],axis=1)) * (np.array(fe_data[fe_cols])*rr_weight).sum(axis=1))  - (np.array(fe_data[fe_cols]).sum(axis=1) * fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[2],axis=1))) 
        # fe_data['rr_trend_d'] = (((13 - fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[1],axis = 1)) * fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[3],axis =1)) -(fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[2],axis =1))**2 )
        # fe_data['rr_trend'] = fe_data['rr_trend_n']/fe_data['rr_trend_d']
        # fe_data['rr_trend'] = fe_data['rr_trend'].fillna(0)
        # fe_data['rr_intent_trend_type'] = fe_data['rr_trend'].apply(lambda x: intent_trend_type(x))
        

        # Notes
        # fe.ge(1.0): This method compares each element in the DataFrame fe to 1.0. 
        # (True if the element is greater than or equal to 1.0, False otherwise).
        # .astype(int): This method converts the boolean values to integers (1 for True and 0 for False).
        fe = fe_data[fe_cols]
        fe = fe.ge(1.0).astype(int)
        
        
        # This multiplies the sum of the weighted values by 1/91 to normalize the result.
        # rr_w13 to rr_w01 || rr_weight 1 to 13
        fe_data['rr_regularity'] = (1/91)*(np.array(fe)*rr_weight).sum(axis=1)
        fe_data['rr_regularity_scaled'] = fe_data['rr_regularity']*fe_data['rr_intent_multiplier']
        
        # Unused
        # fe_data['rr_regularity_type'] = fe_data['rr_regularity_scaled'].apply(lambda x: 'Low' if x <= 0.3 else 'High')
        # fe_data['rr_recency_type'] = fe_data['days_since_last_rr'].apply(lambda x: recency_type(x))
        
        fe_data['active_days_mean'] =  back_up[activedays_cols].mean(axis=1) ## Check logic once
        # fe_data['rr_regularity_final_old'] = fe_data.apply(lambda x: regularity_decider(x['rr_regularity_scaled'],x['active_days_mean']),axis=1)
        
        print('Calctn-2')
        fe_data['rr_regularity_final'] = fe_data.apply(lambda x: regularity_decider_v2(x['rr_regularity_scaled'],x['active_days_mean']),axis=1)
        fe_data['reg_intent'] = fe_data.rr_regularity_final+'_'+fe_data.rr_mean_intent_final
        
        # print('Calctn-3')
        # fe_data['consideration_final'] = fe_data.apply(lambda x: consideration_signal_creator(x['rr_regularity_scaled'],x['rr_mean_intent'],x['active_days_mean']),axis=1)
        # fe_data['rr_intent_trend_type_v2']= fe_data.apply(lambda x: custom_func(x['rr_mean_intent_type'],x['rr_intent_trend_type']),axis=1)

        return(fe_data)

    
    print('UDF-Created Successfully')
    
    
    L3_signals = create_L3_signals(fe_data,back_up)
    L3_signals['city'] = city_name
    
    final_data = L3_signals.copy()
    
    # final_data = final_data.add_suffix('_nov22')
    # final_data['target']=np.where(final_data.next_lts=='DORMANT',1,0)
    
    return final_data    

## Signal Generation

In [5]:
concatenated_df=pd.DataFrame()

for i in date_list:
    
    print(i)
    tdf = consideration_query(i, city, base_week)
    
    print(tdf.shape[0])
    # concatenated_df=concatenated_df.append(tdf).reset_index(drop=True)
    concatenated_df=pd.concat([concatenated_df, tdf]).reset_index(drop=True)

2024-04-07
(460639, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
444026
2024-04-14
(460639, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
445640
2024-04-21
(460639, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
444137
2024-04-28
(460639, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
444295
2024-05-05
(460639, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
444427


In [27]:
concatenated_df.rr_regularity_scaled.quantile([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0])

0.00    0.010989
0.10    0.208791
0.20    0.296703
0.30    0.384615
0.40    0.472527
0.50    0.560440
0.60    0.648352
0.70    0.747253
0.80    0.857143
0.90    0.978022
0.95    1.000000
0.99    1.000000
1.00    1.000000
Name: rr_regularity_scaled, dtype: float64

In [28]:
concatenated_df.active_days_mean.quantile([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0])

0.00    1.000000
0.10    1.000000
0.20    1.000000
0.30    1.250000
0.40    1.400000
0.50    1.555556
0.60    1.750000
0.70    2.000000
0.80    2.400000
0.90    3.181818
0.95    4.000000
0.99    5.333333
1.00    7.000000
Name: active_days_mean, dtype: float64

In [29]:
concatenated_df.rr_mean_intent.quantile([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0])

0.00     1.000000
0.10     1.000000
0.20     1.428571
0.30     1.666667
0.40     2.000000
0.50     2.166667
0.60     2.500000
0.70     3.000000
0.80     3.666667
0.90     5.076923
0.95     6.700000
0.99    10.666667
1.00    58.846154
Name: rr_mean_intent, dtype: float64

In [6]:
concatenated_df['reg_intent'] = concatenated_df.rr_regularity_final + '_' + concatenated_df.rr_mean_intent_final
base_df = concatenated_df[concatenated_df.run_date == base_week].reset_index(drop=True)

In [7]:
concatenated_df.head(3)

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date,rr_intent_multiplier,rr_mean_intent,rr_mean_intent_final,rr_regularity,rr_regularity_scaled,active_days_mean,rr_regularity_final,reg_intent,city
0,628843938f9b43563f5f9868,687,690.0,2024-04-04,2022-05-21,HOOK,HOOK,LOW_INCOME,13,RECENT,4,Chennai,Low,Increasing,High,High,Recent,High,MALE,PS,UNKNOWN,NO_AFFINITY,LONG_DISTANCE,EVENING_PEAK,UNKNOWN,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,628843938f9b43563f5f9868,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,2.0,1.0,1.0,9.0,9.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,1.0,1.0,3.0,43.0,686.0,690.0,2024-04-07,1.0,1.5,03_Low,0.615385,0.615385,1.166667,03_bi_weekly,03_bi_weekly_03_Low,Chennai
1,64f00e674548b9ba9d3606cc,220,220.0,2024-04-05,2023-08-31,SUSTENANCE,DORMANT,HIGH_INCOME,18,RECENT,2,Chennai,Low,Stable,High,High,Recent,High,FEMALE,UNKNOWN,UNKNOWN,ONLY_AUTO,LONG_DISTANCE,MORNING_PEAK,WEEKDAY,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,64f00e674548b9ba9d3606cc,0.0,2.0,1.0,0.0,3.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,2.0,12.0,13.0,0.0,1.0,1.0,0.0,3.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0,2.0,1.0,37.0,219.0,220.0,2024-04-07,1.0,2.0,03_Low,0.417582,0.417582,1.666667,03_bi_weekly,03_bi_weekly_03_Low,Chennai
2,5e47c8de2c8b2a019af38313,248,250.0,2024-04-08,2023-08-03,HOOK,SOFT_CHURN,HIGH_INCOME,10,RECENT,2,Chennai,Low,Stable,High,High,Recent,High,FEMALE,UNKNOWN,NPS,AUTO_CAB,UNKNOWN,EVENING_PEAK,UNKNOWN,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,5e47c8de2c8b2a019af38313,1.0,3.0,1.0,0.0,1.0,2.0,0.0,2.0,0.0,1.0,0.0,0.0,1.0,12.0,12.0,1.0,2.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,5.0,31.0,247.0,250.0,2024-04-07,1.0,1.5,03_Low,0.527473,0.527473,1.125000,03_bi_weekly,03_bi_weekly_03_Low,Chennai


In [8]:
base_df.head(3)

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date,rr_intent_multiplier,rr_mean_intent,rr_mean_intent_final,rr_regularity,rr_regularity_scaled,active_days_mean,rr_regularity_final,reg_intent,city
0,628843938f9b43563f5f9868,687,690.0,2024-04-04,2022-05-21,HOOK,HOOK,LOW_INCOME,13,RECENT,4,Chennai,Low,Increasing,High,High,Recent,High,MALE,PS,UNKNOWN,NO_AFFINITY,LONG_DISTANCE,EVENING_PEAK,UNKNOWN,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,628843938f9b43563f5f9868,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,2.0,1.0,1.0,9.0,9.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,1.0,1.0,3.0,43.0,686.0,690.0,2024-04-07,1.0,1.5,03_Low,0.615385,0.615385,1.166667,03_bi_weekly,03_bi_weekly_03_Low,Chennai
1,64f00e674548b9ba9d3606cc,220,220.0,2024-04-05,2023-08-31,SUSTENANCE,DORMANT,HIGH_INCOME,18,RECENT,2,Chennai,Low,Stable,High,High,Recent,High,FEMALE,UNKNOWN,UNKNOWN,ONLY_AUTO,LONG_DISTANCE,MORNING_PEAK,WEEKDAY,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,64f00e674548b9ba9d3606cc,0.0,2.0,1.0,0.0,3.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,2.0,12.0,13.0,0.0,1.0,1.0,0.0,3.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0,2.0,1.0,37.0,219.0,220.0,2024-04-07,1.0,2.0,03_Low,0.417582,0.417582,1.666667,03_bi_weekly,03_bi_weekly_03_Low,Chennai
2,5e47c8de2c8b2a019af38313,248,250.0,2024-04-08,2023-08-03,HOOK,SOFT_CHURN,HIGH_INCOME,10,RECENT,2,Chennai,Low,Stable,High,High,Recent,High,FEMALE,UNKNOWN,NPS,AUTO_CAB,UNKNOWN,EVENING_PEAK,UNKNOWN,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,5e47c8de2c8b2a019af38313,1.0,3.0,1.0,0.0,1.0,2.0,0.0,2.0,0.0,1.0,0.0,0.0,1.0,12.0,12.0,1.0,2.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,5.0,31.0,247.0,250.0,2024-04-07,1.0,1.5,03_Low,0.527473,0.527473,1.125000,03_bi_weekly,03_bi_weekly_03_Low,Chennai


In [9]:
concatenated_df.to_csv('concatenated_df_old.csv', index=False)
base_df.to_csv('base_df_old.csv', index=False)

# concatenated_df = pd.read_csv('concatenated_df_old.csv')
# base_df = pd.read_csv('base_df_old.csv')

## Stability Function

In [10]:
def stability_func(df,base_week):
    
    df_base_week = df[df.run_date == base_week][['customer','run_date','rr_regularity_final','rr_mean_intent_final','reg_intent']]\
                                                    .reset_index(drop=True)
    ranking = {'05_rare_need' : 1,
               '04_monthly' : 2,
               '03_bi_weekly' : 3,
               '02_weekly' : 4,
               '01_daily' : 5
              }

    intent_ranking = {'03_Low' : 1,
                      '02_Medium' : 2,
                      '01_High' : 3
                     }

    final_rank = {'05_rare_need_03_Low' : 1,
                  '05_rare_need_02_Medium' : 2,
                  '05_rare_need_01_High' : 3,
                  '04_monthly_03_Low' : 4,
                  '04_monthly_02_Medium' : 5,
                  '04_monthly_01_High' : 6,
                  '03_bi_weekly_03_Low' : 7,
                  '03_bi_weekly_02_Medium' : 8,
                  '03_bi_weekly_01_High' : 9,
                  '02_weekly_03_Low' : 10,
                  '02_weekly_02_Medium' : 11,
                  '02_weekly_01_High' : 12,
                  '01_daily_02_Medium' : 13,
                  '01_daily_01_High' : 14
                 }    

    df['reg_rank'] = df['rr_regularity_final'].map(ranking)
    df['int_rank'] = df['rr_mean_intent_final'].map(intent_ranking)
    df['final_rank'] = df['reg_intent'].map(final_rank)

    
    
    ## Regularity Stabillity
    pvt_df = pd.pivot_table(df, values='reg_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()
    pvt_df.columns
    dropped_reg = pvt_df.iloc[:, 1:].lt(pvt_df[base_week], axis=0)

    # Display the result
    print(dropped_reg.head())
    dropped_reg.head()
    first_dropped_week = dropped_reg.idxmax(axis=1)

    pvt_df['first_dropped_week'] = first_dropped_week.where(dropped_reg.any(axis=1), 'Maintained/Upgraded')

    pvt_df.reset_index(inplace=True)
    pvt_df.head()

    df_base_week_stability = pvt_df.merge(df_base_week, on=['customer'], how='inner')

    reg_stab = df_base_week_stability\
                        .groupby(by=['run_date','rr_regularity_final','first_dropped_week'], as_index=False)\
                        .agg(user_count=('customer','nunique'))\
                        .pivot_table(values='user_count', 
                                     index=['run_date','rr_regularity_final',], 
                                     columns='first_dropped_week', 
                                     aggfunc='sum')\
                        .reset_index()\
                        .merge(df_base_week\
                                   .groupby(by=['rr_regularity_final'],as_index=False)\
                                   .agg(base=('customer','nunique')), on=['rr_regularity_final'], how='inner')\
                        .fillna(0)
    
    # .to_csv(f'{month}/output_files/regularity_stability.csv',index=False)

    
    
    
    ## Intent Stabillity
    pvt_df_int=pd.pivot_table(df, values='int_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()
    pvt_df_int.columns
    dropped_int = pvt_df_int.iloc[:, 1:].lt(pvt_df_int[base_week], axis=0)

    # Display the result
    print(dropped_int.head())
    dropped_int.head()
    int_first_dropped_week = dropped_int.idxmax(axis=1)

    pvt_df_int['int_first_dropped_week'] = int_first_dropped_week.where(dropped_int.any(axis=1), 'Maintained/Upgraded')

    pvt_df_int.reset_index(inplace=True)
    pvt_df_int.head()

    df_base_week_int_stability = pvt_df_int.merge(df_base_week, on=['customer'], how='inner')

    int_stab = df_base_week_int_stability\
                        .groupby(by=['run_date','rr_mean_intent_final','int_first_dropped_week'],as_index=False)\
                        .agg(user_count=('customer','nunique'))\
                        .pivot_table(values='user_count', 
                                     index=['run_date','rr_mean_intent_final',], 
                                     columns='int_first_dropped_week', 
                                     aggfunc='sum')\
                        .reset_index()\
                        .merge(df_base_week\
                                    .groupby(by=['rr_mean_intent_final'], as_index=False)\
                                    .agg(base=('customer','nunique')), on=['rr_mean_intent_final'], how='inner')\
                        .fillna(0)
    
    # .to_csv(f'{month}/output_files/intent_stability.csv',index=False)

    
    
    
    ## Final Stabillity
    pvt_df_final=pd.pivot_table(df, values='final_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()
    pvt_df_final.columns
    dropped_final = pvt_df_final.iloc[:, 1:].lt(pvt_df_final[base_week], axis=0)

    
    # Display the result
    print(dropped_final.head())
    dropped_final.head()
    final_first_dropped_week = dropped_final.idxmax(axis=1)

    pvt_df_final['final_first_dropped_week'] = final_first_dropped_week.where(dropped_final.any(axis=1), 'Maintained/Upgraded')

    pvt_df_final.reset_index(inplace=True)
    pvt_df_final.head()

    df_base_week_final_stability = pvt_df_final.merge(df_base_week, on=['customer'], how='inner')

    final_stab = df_base_week_final_stability\
                        .groupby(by=['run_date','rr_regularity_final','rr_mean_intent_final','final_first_dropped_week'],
                                 as_index=False)\
                        .agg(user_count=('customer','nunique'))\
                        .pivot_table(values='user_count', 
                                     index=['run_date','rr_regularity_final','rr_mean_intent_final'], 
                                     columns='final_first_dropped_week', 
                                     aggfunc='sum')\
                        .reset_index()\
                        .merge(df_base_week\
                                   .groupby(by=['rr_regularity_final','rr_mean_intent_final'], as_index=False)\
                                   .agg(base=('customer','nunique')), on=['rr_regularity_final','rr_mean_intent_final'], how='inner')\
                        .fillna(0)
    
    # .to_csv(f'{month}/output_files/final_stability.csv',index=False)
    
    return reg_stab, int_stab, final_stab

In [11]:
reg_df, int_df, final_seg_df = stability_func(concatenated_df, base_week)

run_date  2024-04-07  2024-04-14  2024-04-21  2024-04-28  2024-05-05
0              False       False       False       False       False
1              False       False       False       False       False
2              False       False       False       False       False
3              False       False       False       False       False
4              False       False       False       False       False
run_date  2024-04-07  2024-04-14  2024-04-21  2024-04-28  2024-05-05
0              False       False       False       False       False
1              False       False       False       False       False
2              False       False       False       False       False
3              False       False        True        True        True
4              False       False       False       False       False
run_date  2024-04-07  2024-04-14  2024-04-21  2024-04-28  2024-05-05
0              False       False       False       False       False
1              False       False  

In [12]:
# complete flow
def stab_func_2(df, base_week, base_df, cons_type):
    
    # base_df=df[df.run_date==base_week].reset_index(drop=True)
    fdf1 = base_df.groupby(by=['city', 'run_date', cons_type], as_index=False).agg(base=('customer','nunique'))
    j = 0
    
    for i in date_list:
        
        j += 1
        if i != base_week:
            
            tdf = df[df.run_date==base_week][['run_date', 'customer', cons_type]]\
                                                .merge(df[df.run_date == i][['customer', cons_type]],on=['customer'],how='left')\
                                                .pivot_table(values='customer', 
                                                             index=['run_date',cons_type+'_x'], 
                                                             columns=cons_type+'_y', aggfunc='nunique')\
                                                .reset_index()\
                                                .fillna(0)
            
            # print(list(tdf))
            items_to_remove = ['run_date', cons_type+'_x']
            filtered_list = [x for x in list(tdf) if x not in items_to_remove]
            tdf.columns = ['run_date', cons_type]+[ j + '_' + i for j in filtered_list]
            fdf1 = fdf1.merge(tdf, on=['run_date', cons_type], how='inner')
    
    return fdf1
        
        

In [13]:
#cons_type: 'rr_regularity_final', 'rr_mean_intent_final', 'reg_intent'
finaldf = stab_func_2(concatenated_df, base_week, base_df, 'reg_intent')

In [14]:
finaldf

,city,run_date,reg_intent,base,01_daily_01_High_2024-04-14,01_daily_02_Medium_2024-04-14,02_weekly_01_High_2024-04-14,02_weekly_02_Medium_2024-04-14,02_weekly_03_Low_2024-04-14,03_bi_weekly_01_High_2024-04-14,03_bi_weekly_02_Medium_2024-04-14,03_bi_weekly_03_Low_2024-04-14,04_monthly_01_High_2024-04-14,04_monthly_02_Medium_2024-04-14,04_monthly_03_Low_2024-04-14,05_rare_need_01_High_2024-04-14,05_rare_need_02_Medium_2024-04-14,05_rare_need_03_Low_2024-04-14,01_daily_01_High_2024-04-21,01_daily_02_Medium_2024-04-21,02_weekly_01_High_2024-04-21,02_weekly_02_Medium_2024-04-21,02_weekly_03_Low_2024-04-21,03_bi_weekly_01_High_2024-04-21,03_bi_weekly_02_Medium_2024-04-21,03_bi_weekly_03_Low_2024-04-21,04_monthly_01_High_2024-04-21,04_monthly_02_Medium_2024-04-21,04_monthly_03_Low_2024-04-21,05_rare_need_01_High_2024-04-21,05_rare_need_02_Medium_2024-04-21,05_rare_need_03_Low_2024-04-21,01_daily_01_High_2024-04-28,01_daily_02_Medium_2024-04-28,02_weekly_01_High_2024-04-28,02_weekly_02_Medium_2024-04-28,02_weekly_03_Low_2024-04-28,03_bi_weekly_01_High_2024-04-28,03_bi_weekly_02_Medium_2024-04-28,03_bi_weekly_03_Low_2024-04-28,04_monthly_01_High_2024-04-28,04_monthly_02_Medium_2024-04-28,04_monthly_03_Low_2024-04-28,05_rare_need_01_High_2024-04-28,05_rare_need_02_Medium_2024-04-28,05_rare_need_03_Low_2024-04-28,01_daily_01_High_2024-05-05,01_daily_02_Medium_2024-05-05,02_weekly_01_High_2024-05-05,02_weekly_02_Medium_2024-05-05,02_weekly_03_Low_2024-05-05,03_bi_weekly_01_High_2024-05-05,03_bi_weekly_02_Medium_2024-05-05,03_bi_weekly_03_Low_2024-05-05,04_monthly_01_High_2024-05-05,04_monthly_02_Medium_2024-05-05,04_monthly_03_Low_2024-05-05,05_rare_need_01_High_2024-05-05,05_rare_need_02_Medium_2024-05-05,05_rare_need_03_Low_2024-05-05
0,Chennai,2024-04-07,01_daily_01_High,35910,32798.0,1757.0,214.0,320.0,0.0,789.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,31444.0,2137.0,233.0,452.0,0.0,1175.0,218.0,0.0,251.0,0.0,0.0,0.0,0.0,0.0,29995.0,2494.0,238.0,715.0,0.0,1745.0,398.0,0.0,325.0,0.0,0.0,0.0,0.0,0.0,28280.0,2854.0,219.0,1006.0,1.0,2254.0,629.0,2.0,343.0,113.0,0.0,209.0,0.0,0.0
1,Chennai,2024-04-07,01_daily_02_Medium,24356,1630.0,19272.0,3.0,2528.0,11.0,16.0,896.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2544.0,17066.0,8.0,3058.0,12.0,77.0,1455.0,10.0,0.0,126.0,0.0,0.0,0.0,0.0,3465.0,14866.0,11.0,3526.0,23.0,155.0,2113.0,10.0,0.0,187.0,0.0,0.0,0.0,0.0,3945.0,13052.0,14.0,3936.0,35.0,218.0,2688.0,20.0,23.0,314.0,9.0,0.0,102.0,0.0
2,Chennai,2024-04-07,02_weekly_01_High,1425,210.0,6.0,639.0,321.0,0.0,242.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,271.0,9.0,457.0,329.0,1.0,154.0,81.0,0.0,123.0,0.0,0.0,0.0,0.0,0.0,287.0,10.0,350.0,341.0,0.0,165.0,126.0,1.0,145.0,0.0,0.0,0.0,0.0,0.0,278.0,18.0,276.0,348.0,0.0,140.0,133.0,2.0,72.0,48.0,0.0,110.0,0.0,0.0
3,Chennai,2024-04-07,02_weekly_02_Medium,72971,339.0,2505.0,273.0,59235.0,2225.0,15.0,8067.0,312.0,0.0,0.0,0.0,0.0,0.0,0.0,688.0,3724.0,311.0,53131.0,2684.0,56.0,11009.0,843.0,0.0,525.0,0.0,0.0,0.0,0.0,1163.0,4791.0,343.0,48323.0,3216.0,106.0,12673.0,1400.0,0.0,950.0,6.0,0.0,0.0,0.0,1581.0,5317.0,377.0,43386.0,3240.0,157.0,14352.0,2161.0,30.0,1714.0,191.0,0.0,465.0,0.0
4,Chennai,2024-04-07,02_weekly_03_Low,29730,12.0,25.0,12.0,2906.0,18622.0,0.0,179.0,7974.0,0.0,0.0,0.0,0.0,0.0,0.0,19.0,34.0,3.0,4004.0,14973.0,10.0,705.0,8188.0,0.0,0.0,1794.0,0.0,0.0,0.0,25.0,69.0,5.0,5055.0,12626.0,10.0,1093.0,8888.0,0.0,1.0,1958.0,0.0,0.0,0.0,27.0,109.0,4.0,5430.0,9998.0,4.0,1551.0,9068.0,9.0,144.0,1774.0,0.0,0.0,1612.0
5,Chennai,2024-04-07,03_bi_weekly_01_High,3557,700.0,72.0,56.0,57.0,0.0,2095.0,391.0,1.0,167.0,18.0,0.0,0.0,0.0,0.0,1042.0,127.0,70.0,112.0,0.0,1427.0,478.0,2.0,259.0,39.0,1.0,0.0,0.0,0.0,1148.0,149.0,69.0,160.0,0.0,1048.0,545.0,5.0,340.0,88.0,3.0,1.0,1.0,0.0,1131.0,191.0,65.0,218.0,1.0,804.0,522.0,23.0,402.0,151.0,12.0,24.0,11.0,2.0
6,Chennai,2024-04-07,03_bi_weekly_02_Medium,55273,101.0,1023.0,27.0,7666.0,641.0,354.0,37788.0,3415.0,6.0,3894.0,358.0,0.0,0.0,0.0,351.0,1517.0,48.0,10241.0,1091.0,413.0,30808.0,4618.0,25.0,547

In [15]:
base_df\
.groupby(by=['city_name','run_date','reg_intent'],as_index=False)\
.agg(
    user_count=('customer','nunique'),
    fe_retention_w01=('fe_flag_w01','sum'),
    fe_retention_w02=('fe_flag_w02','sum'),
    fe_retention_w03=('fe_flag_w03','sum'),
    fe_retention_w04=('fe_flag_w04','sum'),
    fe_retention_w05=('fe_flag_w05','sum'),
    fe_retention_w06=('fe_flag_w06','sum'),
    fe_retention_w07=('fe_flag_w07','sum'),
    fe_retention_w08=('fe_flag_w08','sum'),
    fe_retention_w09=('fe_flag_w09','sum'),
    fe_retention_w10=('fe_flag_w10','sum'),
    
    rr_retention_w01=('rr_flag_w01','sum'),
    rr_retention_w02=('rr_flag_w02','sum'),
    rr_retention_w03=('rr_flag_w03','sum'),
    rr_retention_w04=('rr_flag_w04','sum'),
    rr_retention_w05=('rr_flag_w05','sum'),
    rr_retention_w06=('rr_flag_w06','sum'),
    rr_retention_w07=('rr_flag_w07','sum'),
    rr_retention_w08=('rr_flag_w08','sum'),
    rr_retention_w09=('rr_flag_w09','sum'),
    rr_retention_w10=('rr_flag_w10','sum'),
        
    nr_retention_w01=('nr_flag_w01','sum'),
    nr_retention_w02=('nr_flag_w02','sum'),
    nr_retention_w03=('nr_flag_w03','sum'),
    nr_retention_w04=('nr_flag_w04','sum'),
    nr_retention_w05=('nr_flag_w05','sum'),
    nr_retention_w06=('nr_flag_w06','sum'),
    nr_retention_w07=('nr_flag_w07','sum'),
    nr_retention_w08=('nr_flag_w08','sum'),
    nr_retention_w09=('nr_flag_w09','sum'),
    nr_retention_w10=('nr_flag_w10','sum')
    )


# .to_csv(f'{month}/output_files/reg_retention_10w_newlogic.csv',index=False)

#rr_mean_intent_final, rr_regularity_final, reg_intent

# gender_tag,
# taxi_income_segment,
# ps_tag_link,
# ps_tag_auto,
# taxi_service_affinity,
# taxi_distance_preference,
# taxi_time_of_day_preference,
# taxi_day_of_week_preference

,city_name,run_date,reg_intent,user_count,fe_retention_w01,fe_retention_w02,fe_retention_w03,fe_retention_w04,fe_retention_w05,fe_retention_w06,fe_retention_w07,fe_retention_w08,fe_retention_w09,fe_retention_w10,rr_retention_w01,rr_retention_w02,rr_retention_w03,rr_retention_w04,rr_retention_w05,rr_retention_w06,rr_retention_w07,rr_retention_w08,rr_retention_w09,rr_retention_w10,nr_retention_w01,nr_retention_w02,nr_retention_w03,nr_retention_w04,nr_retention_w05,nr_retention_w06,nr_retention_w07,nr_retention_w08,nr_retention_w09,nr_retention_w10
0,Chennai,2024-04-07,01_daily_01_High,35910,32845,34397,34808,34993,35100,35176,35216,35256,35302,35326,32226,34065,34590,34822,34946,35037,35081,35137,35191,35217,31530,33698,34348,34631,34789,34901,34964,35030,35094,35128
1,Chennai,2024-04-07,01_daily_02_Medium,24356,21432,23031,23508,23712,23796,23871,23915,23946,23977,24006,20716,22653,23259,23532,23649,23738,23799,23838,23873,23913,19807,22169,22959,23330,23499,23613,23693,23746,23781,23829
2,Chennai,2024-04-07,02_weekly_01_High,1425,1033,1160,1206,1230,1243,1254,1259,1263,1268,1271,976,1123,1175,1204,1218,1234,1240,1244,1249,1256,936,1096,1151,1182,1198,1217,1226,1230,1235,1244
3,Chennai,2024-04-07,02_weekly_02_Medium,72971,57177,66157,68915,69992,70592,70923,71114,71284,71413,71515,53263,63893,67491,69015,69836,70315,70588,70829,71007,71154,50003,61475,65836,67824,68903,69569,69957,70276,70535,70741
4,Chennai,2024-04-07,02_weekly_03_Low,29730,18995,23915,25824,26625,27083,27336,27498,27643,27757,27844,16819,22233,24563,25648,26296,26659,26896,27093,27266,27391,15385,20826,23390,24731,25545,26021,26348,26626,26847,26995
5,Chennai,2024-04-07,03_bi_weekly_01_High,3557,2664,2996,3172,3250,3306,3337,3357,3375,3391,3401,2486,2887,3100,3197,3253,3287,3307,3330,3354,3367,2380,2814,3036,3152,3217,3254,3278,3308,3329,3345
6,Chennai,2024-04-07,03_bi_weekly_02_Medium,55273,36472,45569,49305,51178,52197,52824,53193,53495,53737,53898,32341,42387,46987,49385,50781,51652,52227,52661,53035,53307,29850,39850,44845,47654,49364,50464,51210,51788,52267,52622
7,Chennai,2024-04-07,03_bi_weekly_03_Low,76525,41509,57278,64358,68079,70301,71698,72567,73189,73728,74164,34514,50645,58927,63599,66608,68595,69871,70881,71778,72477,30702,46070,54741,59910,63456,65829,67497,68808,69881,70754
8,Chennai,2024-04-07,04_monthly_01_High,1660,1075,1229,1315,1350,1395,1421,1443,1465,1478,1491,983,1166,1270,1314,1361,1390,1413,1438,1456,1474,961,1139,1256,1306,1358,1383,1412,1439,1458,1476
9,Chennai,2024-04-07,04_monthly_02_Medium,23063,12032,15964,17855,19044,19863,20387,20729,21012,21247,21445,10097,14133,16283,17668,18669,19322,19746,20128,20473,20758,9327,13155,15343,16822,17908,18641,19118,19542,19914,20247


In [16]:
test=list(finaldf)

In [17]:
# test.remove([['city','01_daily_2023-11-11']])

In [18]:
test=['city']+[i+'_yo' for i in test]

In [19]:
test

['city',
 'city_yo',
 'run_date_yo',
 'reg_intent_yo',
 'base_yo',
 '01_daily_01_High_2024-04-14_yo',
 '01_daily_02_Medium_2024-04-14_yo',
 '02_weekly_01_High_2024-04-14_yo',
 '02_weekly_02_Medium_2024-04-14_yo',
 '02_weekly_03_Low_2024-04-14_yo',
 '03_bi_weekly_01_High_2024-04-14_yo',
 '03_bi_weekly_02_Medium_2024-04-14_yo',
 '03_bi_weekly_03_Low_2024-04-14_yo',
 '04_monthly_01_High_2024-04-14_yo',
 '04_monthly_02_Medium_2024-04-14_yo',
 '04_monthly_03_Low_2024-04-14_yo',
 '05_rare_need_01_High_2024-04-14_yo',
 '05_rare_need_02_Medium_2024-04-14_yo',
 '05_rare_need_03_Low_2024-04-14_yo',
 '01_daily_01_High_2024-04-21_yo',
 '01_daily_02_Medium_2024-04-21_yo',
 '02_weekly_01_High_2024-04-21_yo',
 '02_weekly_02_Medium_2024-04-21_yo',
 '02_weekly_03_Low_2024-04-21_yo',
 '03_bi_weekly_01_High_2024-04-21_yo',
 '03_bi_weekly_02_Medium_2024-04-21_yo',
 '03_bi_weekly_03_Low_2024-04-21_yo',
 '04_monthly_01_High_2024-04-21_yo',
 '04_monthly_02_Medium_2024-04-21_yo',
 '04_monthly_03_Low_2024-04-2

In [30]:
finaldf.to_csv('rr_stability_v0_old.csv', index=False)

In [31]:
finaldf.head(2)

,city,run_date,reg_intent,base,01_daily_01_High_2024-04-14,01_daily_02_Medium_2024-04-14,02_weekly_01_High_2024-04-14,02_weekly_02_Medium_2024-04-14,02_weekly_03_Low_2024-04-14,03_bi_weekly_01_High_2024-04-14,03_bi_weekly_02_Medium_2024-04-14,03_bi_weekly_03_Low_2024-04-14,04_monthly_01_High_2024-04-14,04_monthly_02_Medium_2024-04-14,04_monthly_03_Low_2024-04-14,05_rare_need_01_High_2024-04-14,05_rare_need_02_Medium_2024-04-14,05_rare_need_03_Low_2024-04-14,01_daily_01_High_2024-04-21,01_daily_02_Medium_2024-04-21,02_weekly_01_High_2024-04-21,02_weekly_02_Medium_2024-04-21,02_weekly_03_Low_2024-04-21,03_bi_weekly_01_High_2024-04-21,03_bi_weekly_02_Medium_2024-04-21,03_bi_weekly_03_Low_2024-04-21,04_monthly_01_High_2024-04-21,04_monthly_02_Medium_2024-04-21,04_monthly_03_Low_2024-04-21,05_rare_need_01_High_2024-04-21,05_rare_need_02_Medium_2024-04-21,05_rare_need_03_Low_2024-04-21,01_daily_01_High_2024-04-28,01_daily_02_Medium_2024-04-28,02_weekly_01_High_2024-04-28,02_weekly_02_Medium_2024-04-28,02_weekly_03_Low_2024-04-28,03_bi_weekly_01_High_2024-04-28,03_bi_weekly_02_Medium_2024-04-28,03_bi_weekly_03_Low_2024-04-28,04_monthly_01_High_2024-04-28,04_monthly_02_Medium_2024-04-28,04_monthly_03_Low_2024-04-28,05_rare_need_01_High_2024-04-28,05_rare_need_02_Medium_2024-04-28,05_rare_need_03_Low_2024-04-28,01_daily_01_High_2024-05-05,01_daily_02_Medium_2024-05-05,02_weekly_01_High_2024-05-05,02_weekly_02_Medium_2024-05-05,02_weekly_03_Low_2024-05-05,03_bi_weekly_01_High_2024-05-05,03_bi_weekly_02_Medium_2024-05-05,03_bi_weekly_03_Low_2024-05-05,04_monthly_01_High_2024-05-05,04_monthly_02_Medium_2024-05-05,04_monthly_03_Low_2024-05-05,05_rare_need_01_High_2024-05-05,05_rare_need_02_Medium_2024-05-05,05_rare_need_03_Low_2024-05-05
0,Chennai,2024-04-07,01_daily_01_High,35910,32798.0,1757.0,214.0,320.0,0.0,789.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,31444.0,2137.0,233.0,452.0,0.0,1175.0,218.0,0.0,251.0,0.0,0.0,0.0,0.0,0.0,29995.0,2494.0,238.0,715.0,0.0,1745.0,398.0,0.0,325.0,0.0,0.0,0.0,0.0,0.0,28280.0,2854.0,219.0,1006.0,1.0,2254.0,629.0,2.0,343.0,113.0,0.0,209.0,0.0,0.0
1,Chennai,2024-04-07,01_daily_02_Medium,24356,1630.0,19272.0,3.0,2528.0,11.0,16.0,896.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2544.0,17066.0,8.0,3058.0,12.0,77.0,1455.0,10.0,0.0,126.0,0.0,0.0,0.0,0.0,3465.0,14866.0,11.0,3526.0,23.0,155.0,2113.0,10.0,0.0,187.0,0.0,0.0,0.0,0.0,3945.0,13052.0,14.0,3936.0,35.0,218.0,2688.0,20.0,23.0,314.0,9.0,0.0,102.0,0.0


In [32]:
finaldf.to_clipboard(index=False)

## Notes

### Explanation of correction_factor Function

The correction_factor function takes two parameters: rr_weight and rapido_age.

Parameters:

rr_weight: A series or array of weights (assumed to be a pandas Series based on the usage).
rapido_age: An integer representing the age in days.

Logic and Calculations:
The function performs different calculations based on the value of rapido_age.

    Condition 1: rapido_age >= 91:
        week_inactive is set to 0.
        multiplier is set to 1.
        This means there is no inactivity, and the multiplier is 1 (no correction).
    
    Condition 2: rapido_age == 0:
        week_inactive is calculated as (91 - rapido_age) // 7 - 1.
        multiplier is calculated using the formula:
            multiplier = rr_weight.sum() / (rr_weight.sum() - (week_inactive * (week_inactive + 1) / 2))
        This corrects for the inactivity based on the number of weeks inactive.
    
    Condition 3: rapido_age % 7 == 0:
        week_inactive is calculated similarly to Condition 2.
        multiplier is calculated using the same formula as in Condition 2.
    
    Condition 4: All Other Cases:
        week_inactive is calculated as (91 - rapido_age) // 7.
        multiplier is calculated using the same formula as in Conditions 2 and 3.
        
    Return Values:
        The function returns a tuple containing:
            multiplier: The correction factor.
            week_inactive: Number of inactive weeks.
            Corrected sum of weights: rr_weight.sum() - (week_inactive * (week_inactive + 1) / 2).
            Corrected sum of squared weights: (rr_weight * rr_weight).sum() - (week_inactive * (week_inactive + 1) * (2 * week_inactive + 1) / 6).

                
Applying correction_factor to fe_data
    The line fe_data['rr_intent_multiplier'] = fe_data['rapido_age'].apply(lambda x: correction_factor(rr_weight, x)[0]) adds a new column rr_intent_multiplier to the DataFrame fe_data.

- For each value in the column rapido_age of fe_data, the correction_factor function is called.
- The first element of the returned tuple (i.e., multiplier) is extracted and assigned to the new column rr_intent_multiplier.

In [22]:
rapido_age = 0
rr_weight = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])

rr_weight_sum = rr_weight.sum()
week_inactive = (91 - rapido_age) // 7 - 1
correction = (week_inactive * (week_inactive + 1)) // 2
adjusted_sum = rr_weight_sum - correction
multiplier = rr_weight_sum / adjusted_sum

print("rr_weight_sum:", rr_weight_sum)
print("week_inactive:", week_inactive)
print("correction:", correction)
print("adjusted_sum:", adjusted_sum)
print("multiplier:", multiplier)

rr_weight_sum: 91
week_inactive: 12
correction: 78
adjusted_sum: 13
multiplier: 7.0


Explanation of Logic

week_inactive: Number of weeks of inactivity calculated based on rapido_age.

correction: Sum of an arithmetic series representing the cumulative impact of inactive weeks.

adjusted_sum: The total sum of rr_weight after adjusting for inactivity.

multiplier: The correction factor used to adjust the weights considering inactivity.

This correction factor ensures that the impact of inactive weeks is accounted for by adjusting the total weight proportionally.

In [24]:
import numpy as np

rr_weight = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])
rr_weight_sum = rr_weight.sum()

rapido_ages = list(range(0, 91, 5))  # List of rapido_age values from 0 to 90 in increments of 5

for rapido_age in rapido_ages:
    if rapido_age >= 91:
        week_inactive = 0
        correction = 0
    else:
        week_inactive = (91 - rapido_age) // 7 - 1
        correction = np.log((week_inactive * (week_inactive + 1)) // 2)

    adjusted_sum = rr_weight_sum - correction
    multiplier = rr_weight_sum / adjusted_sum

    print(f"rapido_age: {rapido_age}")
    print("week_inactive:", week_inactive)
    print("correction:", correction)
    print("adjusted_sum:", adjusted_sum)
    print("multiplier:", multiplier)
    print("-" * 40)

rapido_age: 0
week_inactive: 12
correction: 4.356708826689592
adjusted_sum: 86.6432911733104
multiplier: 1.0502832794979473
----------------------------------------
rapido_age: 5
week_inactive: 11
correction: 4.189654742026425
adjusted_sum: 86.81034525797358
multiplier: 1.048262159649015
----------------------------------------
rapido_age: 10
week_inactive: 10
correction: 4.007333185232471
adjusted_sum: 86.99266681476753
multiplier: 1.0460651837903214
----------------------------------------
rapido_age: 15
week_inactive: 9
correction: 3.8066624897703196
adjusted_sum: 87.19333751022968
multiplier: 1.0436577220056946
----------------------------------------
rapido_age: 20
week_inactive: 9
correction: 3.8066624897703196
adjusted_sum: 87.19333751022968
multiplier: 1.0436577220056946
----------------------------------------
rapido_age: 25
week_inactive: 8
correction: 3.58351893845611
adjusted_sum: 87.41648106154389
multiplier: 1.040993630662543
----------------------------------------
rapid

Interpretation

rapido_age = 0: The maximum number of inactive weeks (12 weeks) results in a correction of 78, leading to a high multiplier of 7.0.

rapido_age = 5: The number of inactive weeks decreases to 11, with a correction of 66, resulting in a multiplier of approximately 3.64.

rapido_age = 10 to 90: As rapido_age increases, the number of inactive weeks and the corresponding correction decrease, leading to lower multipliers.

For rapido_age values from 80 to 90, the multiplier becomes 1.0, indicating no correction is needed since there are no inactive weeks.

## rr_regularity_final

In [33]:
#cons_type: 'rr_regularity_final', 'rr_mean_intent_final', 'reg_intent'
finaldf_v2 = stab_func_2(concatenated_df, base_week, base_df, 'rr_regularity_final')

In [34]:
finaldf_v2

,city,run_date,rr_regularity_final,base,01_daily_2024-04-14,02_weekly_2024-04-14,03_bi_weekly_2024-04-14,04_monthly_2024-04-14,05_rare_need_2024-04-14,01_daily_2024-04-21,02_weekly_2024-04-21,03_bi_weekly_2024-04-21,04_monthly_2024-04-21,05_rare_need_2024-04-21,01_daily_2024-04-28,02_weekly_2024-04-28,03_bi_weekly_2024-04-28,04_monthly_2024-04-28,05_rare_need_2024-04-28,01_daily_2024-05-05,02_weekly_2024-05-05,03_bi_weekly_2024-05-05,04_monthly_2024-05-05,05_rare_need_2024-05-05
0,Chennai,2024-04-07,01_daily,60266,55457.0,3076.0,1733.0,0.0,0.0,53191.0,3763.0,2935.0,377.0,0.0,50820.0,4513.0,4421.0,512.0,0.0,48131.0,5211.0,5811.0,802.0,311.0
1,Chennai,2024-04-07,02_weekly,104126,3097.0,84233.0,16796.0,0.0,0.0,4745.0,75893.0,21046.0,2442.0,0.0,6345.0,70259.0,24462.0,3060.0,0.0,7330.0,63059.0,27568.0,3982.0,2187.0
2,Chennai,2024-04-07,03_bi_weekly,135355,1900.0,15221.0,102925.0,15309.0,0.0,3066.0,20703.0,90189.0,21397.0,0.0,3952.0,24666.0,79926.0,26786.0,25.0,4458.0,27248.0,69991.0,32460.0,1198.0
3,Chennai,2024-04-07,04_monthly,96919,0.0,0.0,22838.0,67352.0,6729.0,0.0,0.0,28655.0,55774.0,12490.0,279.0,391.0,31739.0,47044.0,17466.0,812.0,2355.0,31637.0,40033.0,22082.0
4,Chennai,2024-04-07,05_rare_need,47360,0.0,0.0,0.0,15603.0,29758.0,0.0,0.0,969.0,18252.0,24341.0,0.0,0.0,3011.0,18681.0,21717.0,0.0,0.0,5145.0,18218.0,19939.0


In [35]:
finaldf_v2.to_csv('rr_stability_v0_old_agg.csv', index=False)

In [37]:
finaldf_v2.to_clipboard(index=False)